# 1. SERP Data Collection
The function of this notebook is to iteratively parse through a sample of United States cities and retrieve the first page of the Google Search Result Page (SERP) with the query 'Jobs Near Me'. Each iteration of the collection will return a structured .csv results page, a .json results page, and an .html of the webpage. As the entire .html has been saved, it can be reopened on a Google Chrome Web Browser.


**GitRonald: WebSearcher**
* [GitHub Repository](https://github.com/gitronald/WebSearcher?tab=readme-ov-file#example-search-script)
* [Location Based Searching](https://gist.github.com/gitronald/45bad10ca2b78cf4ec1197b542764e05)


---
## 1. Environment Creation

### 1.1 Library Import

In [2]:
''' DATA MANAGEMENT'''
import pandas as pd
import os
import regex as re

''' DATA COLLECTION'''
import WebSearcher as ws
import selenium

''' TIME MANAGEMENT '''
import time
from tqdm.auto import tqdm
import random

/Users/nataliecastro/Library/CloudStorage/OneDrive-Personal/research/job seeking behavior + alg audit/alg audit/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1.2 Data Import

[LOCATION DESC]

In [6]:
''' CANONICAL SEARCH LOCATIONS RECOGNIZED BY GOOGLE '''

locations_dir = '/Users/nataliecastro/Library/CloudStorage/OneDrive-Personal/research/job seeking behavior + alg audit/alg audit/data/locations' 
f = os.listdir(locations_dir)[-1]
fp = os.path.join(locations_dir, f)

## loading the data and displaying the results
locs = pd.read_csv(fp)
locs_df = pd.DataFrame(locs)

## Filtering for United States Canonical Names
us_filter = locs_df['Country Code'] == 'US'
usa_locs = locs_df[us_filter]

city_locs = usa_locs[usa_locs['Target Type'] == 'City']

In [7]:
city_locs.head(2)

,Criteria ID,Name,Canonical Name,Parent ID,Country Code,Target Type,Status
10587,1012873,Anchorage,"Anchorage,Alaska,United States",21132.0,US,City,Active
10588,1012874,Anderson,"Anderson,Alaska,United States",21132.0,US,City,Active


## 2. SERP Retrieval

### 2.1 Function Definintion

In [ ]:
''' LOCATION_FILE_NAME(canon_name)
    This function takes a canonical name and replaces the commas with underscores so the files are saved correctly
'''
def location_file_namer(canon_name):
    return (re.sub(",","_", canon_name))



''' CREATING THE FUNCTION TO BE CALLED FROM THE LOOP '''

def location_list_searching(canon_name, query):
    print (f"\r\n📍🔎 | Searching in: {location}")

    ## DATA STORAGE 
    file_location_name = location_file_namer(canon_name)

    data_dir = os.path.join("data", f"jobs-near-me/{file_location_name}")
    fp_serps = os.path.join(data_dir, f'{file_location_name}.json')
    fp_results = os.path.join(data_dir, f'{file_location_name}.json')
    dir_html = os.path.join(data_dir, 'html')
    os.makedirs(dir_html, exist_ok=True)
    os.makedirs(os.path.dirname(fp_serps), exist_ok=True)
    os.makedirs(os.path.dirname(fp_results), exist_ok=True) 

    ## SEARCHING 
    se = ws.SearchEngine(
        method="selenium", 
        selenium_config = {
            "headless": False,
            "use_subprocess": False,
            "driver_executable_path": "",
            "version_main": 144,
        }
    )

    se.search(query, location=canon_name, ai_expand=True)

    ## RESULT PARSING 
    se.parse_results()
    se.save_serp(append_to=fp_serps)        # Save SERP to json (html + metadata)
    se.save_results(append_to=fp_results)   # Save results to json
    se.save_serp(save_dir=dir_html)         # Save SERP html to dir (no metadata)
    results = pd.DataFrame(se.results)
    results_path = os.path.join(data_dir, f"{canon_name}.csv")
    results.to_csv(results_path)

    ## CLOSING THE SEARCHER 
    se.searcher.cleanup()

    ## NAPPING (so Google isn't angry with me)
    time.sleep(random.uniform(0, 2))
    


### 2.2 Looping Through States

In [ ]:
''' SET THE QUERY HERE: '''
qry = "jobs near me"

In [ ]:
full_sample = city_locs['Canonical Name'].to_list()

In [ ]:
for location in full_sample:
    location_list_searching(location, qry)

<hr style=\"border: none; border-top: 2.5px solid #CACBCE;\" />

*Author:* Natalie Castro  
*Last Edit Date:* January 23, 2026  
*GitHub Repository:* https://github.com/NatalieRMCastro/location-serp-scraper